# TrOCR Fine-Tuning From `custom_dataset`

This notebook is a from-scratch guide for training TrOCR on your own handwritten dataset.
It assumes you only have:
- `custom_dataset/images/`
- `custom_dataset/labels.csv`
- a GPU-enabled local Python environment

The notebook will:
1. download the Microsoft handwritten TrOCR model,
2. convert your CSV into train/val/test manifests,
3. fine-tune on GPU,
4. save the final checkpoint with real weight files, and
5. show how to run the app locally with that checkpoint.

## What to put where

Use this file layout:
- `custom_dataset/labels.csv` for your image-to-text labels
- `custom_dataset/images/` for the image files named in the CSV
- `artifacts/base_models/microsoft_trocr_large_handwritten/` for the downloaded Hugging Face base model
- `data/manifests/` for the train/val/test JSONL files
- `artifacts/trocr_finetuned/` or `artifacts/trocr_airdraw/best/` for the final trained checkpoint

A checkpoint is valid only if it contains a weight file such as `model.safetensors` or `pytorch_model.bin`.
Config files like `config.json` and tokenizer files are not enough by themselves.

If you want the backend to use a local checkpoint, put the final weights in one of those artifact folders and set `AIRDRAW_OCR_MODEL` to that directory if needed.

In [ ]:
from pathlib import Path
import csv
import json
import random
import shutil
import subprocess
import sys

import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# This helper checks whether a checkpoint directory actually contains model weights.
WEIGHT_FILES = [
    "model.safetensors",
    "pytorch_model.bin",
    "model.safetensors.index.json",
    "pytorch_model.bin.index.json",
]


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend").exists() and (candidate / "training").exists() and (candidate / "custom_dataset").exists():
            return candidate
    return start


def checkpoint_has_weights(model_dir: Path) -> bool:
    return any((model_dir / name).exists() for name in WEIGHT_FILES)


def checkpoint_report(model_dir: Path):
    present = [name for name in WEIGHT_FILES if (model_dir / name).exists()]
    missing = [name for name in WEIGHT_FILES if name not in present]
    return present, missing


root = find_project_root(Path.cwd().resolve())
labels_csv = root / "custom_dataset" / "labels.csv"
images_dir = root / "custom_dataset" / "images"
base_model_dir = root / "artifacts" / "base_models" / "microsoft_trocr_large_handwritten"
manifests_dir = root / "data" / "manifests"
output_dir = root / "artifacts" / "trocr_finetuned"

print("project_root:", root)
print("labels_csv_exists:", labels_csv.exists())
print("images_dir_exists:", images_dir.exists())
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

for candidate in [base_model_dir, output_dir, root / "artifacts" / "trocr_airdraw" / "best"]:
    present, missing = checkpoint_report(candidate)
    print(f"{candidate} -> exists={candidate.exists()} has_weights={checkpoint_has_weights(candidate)} present={present} missing={missing}")

## Step 1: download the Microsoft handwritten TrOCR model

Run the next cell once.
It will load the base model from Hugging Face and save a local copy under `artifacts/base_models/microsoft_trocr_large_handwritten/`.

That local copy is what you will fine-tune on your GPU.
Keeping it locally also makes it easy to rerun training without downloading again.

In [ ]:
from pathlib import Path

# Download the base model once, then save it locally so training can reuse it offline.
# This creates a real checkpoint directory with weights.

model_id = "microsoft/trocr-large-handwritten"
base_model_dir.mkdir(parents=True, exist_ok=True)

if checkpoint_has_weights(base_model_dir):
    print(f"Base model already has weights: {base_model_dir}")
else:
    processor = TrOCRProcessor.from_pretrained(model_id)
    model = VisionEncoderDecoderModel.from_pretrained(model_id)
    processor.save_pretrained(base_model_dir)
    model.save_pretrained(base_model_dir)
    print(f"Downloaded and saved base model to: {base_model_dir}")

present, missing = checkpoint_report(base_model_dir)
print("present:", present)
print("missing:", missing)

## Step 2: convert `custom_dataset/labels.csv` into train/val/test manifests

The training script expects JSONL files with rows like:
- `{"image": ".../images/file.png", "text": "hello"}`

The next cell reads your CSV, checks that each image exists, shuffles the rows, and writes:
- `data/manifests/train.jsonl`
- `data/manifests/val.jsonl`
- `data/manifests/test.jsonl`

If you add more samples later, rerun the cell to rebuild the splits.

In [ ]:
# Read the custom dataset CSV and write training manifests.
# This keeps the notebook self-contained: you only need the custom_dataset folder.

rows = []
with labels_csv.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        image_name = (row.get("image") or "").strip()
        text = (row.get("text") or "").strip()
        if not image_name or not text:
            continue
        image_path = images_dir / image_name
        if not image_path.exists():
            print(f"Skipping missing image: {image_path}")
            continue
        rows.append({"image": str(image_path), "text": text})

if len(rows) < 3:
    raise SystemExit(
        f"Need at least 3 labeled samples to create train/val/test splits. Found {len(rows)}."
    )

# Shuffle then split the data.
random.Random(42).shuffle(rows)
count = len(rows)
count_test = max(1, int(count * 0.05)) if count >= 20 else 0
count_val = max(1, int(count * 0.10))
if count_val + count_test >= count:
    count_val = 1
    count_test = 0
count_train = count - count_val - count_test

if count_train <= 0:
    raise SystemExit(
        "Not enough samples for a train split after creating validation/test sets. "
        "Add more labeled examples."
    )

train_rows = rows[:count_train]
val_rows = rows[count_train:count_train + count_val]
test_rows = rows[count_train + count_val:]

manifests_dir.mkdir(parents=True, exist_ok=True)

for filename, subset in [
    ("train.jsonl", train_rows),
    ("val.jsonl", val_rows),
    ("test.jsonl", test_rows),
]:
    target = manifests_dir / filename
    with target.open("w", encoding="utf-8") as f:
        for item in subset:
            f.write(json.dumps(item, ensure_ascii=True) + "\n")
    print(f"wrote {len(subset)} rows -> {target}")

print("total samples:", count)
print("train/val/test:", len(train_rows), len(val_rows), len(test_rows))

# Preview the first training row so you can confirm paths and labels look correct.
if train_rows:
    print("sample row:", train_rows[0])

## Step 3: fine-tune on GPU, then verify the saved weights

Run the next code cell after the manifests are written.
It trains from the local base model copy, saves the best checkpoint, and then checks that the output directory contains real weights.

If the best checkpoint looks good, you can copy it into `artifacts/trocr_finetuned/` so the backend can load it locally.
After that, run the app with:

```bash
uvicorn main:app --reload --host 0.0.0.0 --port 8000
```

If you want to force a specific local checkpoint, set `AIRDRAW_OCR_MODEL` to that folder before starting the backend.

In [ ]:
# Step 3: fine-tune TrOCR on your custom dataset.
# This cell uses the local base model copy, then writes the best checkpoint.
# After training, it copies the best weights into artifacts/trocr_finetuned/
# so the backend can load the model locally without needing Hugging Face.

train_manifest = manifests_dir / "train.jsonl"
val_manifest = manifests_dir / "val.jsonl"
test_manifest = manifests_dir / "test.jsonl"

for required in [base_model_dir, train_manifest, val_manifest]:
    if not required.exists():
        raise SystemExit(f"Missing required path: {required}")

command = [
    sys.executable,
    "training/train_trocr.py",
    "--train-manifest", str(train_manifest),
    "--val-manifest", str(val_manifest),
    "--model-name", str(base_model_dir),
    "--output-dir", str(output_dir),
    "--epochs", "5",
    "--train-batch-size", "4",
    "--eval-batch-size", "4",
    "--gradient-accumulation-steps", "2",
    "--learning-rate", "5e-5",
    "--warmup-ratio", "0.1",
    "--max-target-length", "64",
    "--dataloader-num-workers", "2",
    "--save-total-limit", "2",
]

# Add the test manifest only if it actually has rows.
if test_manifest.exists() and test_manifest.stat().st_size > 0:
    command.extend(["--test-manifest", str(test_manifest)])

# Use fp16 on CUDA to save memory and speed up training.
if torch.cuda.is_available():
    command.append("--fp16")

print("Training command:")
print(" ".join(command))
subprocess.run(command, check=True)

best_dir = output_dir / "best"
present, missing = checkpoint_report(best_dir)
print("best checkpoint:", best_dir)
print("present:", present)
print("missing:", missing)

if not checkpoint_has_weights(best_dir):
    raise SystemExit(
        "The best checkpoint does not contain weights. "
        "Check the training output and rerun after fixing the training run."
    )

# Copy the final checkpoint into the deploy folder the backend checks first.
# This makes `artifacts/trocr_finetuned/` a complete offline model directory.
for item in best_dir.iterdir():
    if item.is_file():
        shutil.copy2(item, output_dir / item.name)

present, missing = checkpoint_report(output_dir)
print("deploy folder:", output_dir)
print("present:", present)
print("missing:", missing)
print("If you want the backend to use this folder explicitly, set AIRDRAW_OCR_MODEL to:")
print(str(output_dir))